# Econ Data Pipeline: Delta Lake & Time Travel (Kaggle Version)
This notebook demonstrates how to connect a Kaggle environment directly to an AWS S3 bucket containing Delta Lake tables, query them using PySpark, and utilize Delta's "Time Travel" feature.

## 1. Setup Environment & Install Dependencies
Kaggle comes with many libraries, but we need `pyspark` and the Hadoop-AWS packages to allow Spark to read from S3 directly.

In [ ]:
!pip install -q pyspark

print("Dependencies installed!")

## 2. Configure Spark & AWS S3 Access
**Instructions:**
1. Go to **Add-ons -> Secrets** in the top menu of your Kaggle notebook.
2. Add a new secret named `AWS_ACCESS_KEY_ID` with your key.
3. Add a new secret named `AWS_SECRET_ACCESS_KEY` with your secret.

We inject these straight into the Hadoop config of the PySpark session.

In [ ]:
from kaggle_secrets import UserSecretsClient
from pyspark.sql import SparkSession

# Stop any existing session to ensure clean JAR loading
try:
    spark.stop()
except:
    pass

# 1. Fetch credentials from the Kaggle Add-on
try:
    user_secrets = UserSecretsClient()
    aws_key = user_secrets.get_secret("AWS_ACCESS_KEY_ID") 
    aws_secret = user_secrets.get_secret("AWS_SECRET_ACCESS_KEY")
except Exception as e:
    print("⚠️ Please configure your AWS keys in the Kaggle Secrets menu!")
    raise e

# 2. Define versions (Matched for Spark 4.0.1 in Kaggle)
packages = [
    "io.delta:delta-spark_2.13:4.0.0",
    "org.apache.hadoop:hadoop-aws:3.4.0" 
]

spark = SparkSession.builder \
    .appName("S3DeltaSpark4") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.access.key", aws_key) \
    .config("spark.hadoop.fs.s3a.secret.key", aws_secret) \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-1.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

print(f"Spark initialized, connected to S3. Version: {spark.version}")

## 3. Read the Latest Delta Table
We can now read the Delta data directly from the S3 URI, exactly as it works on a standard cluster.

In [ ]:
S3_BUCKET = "econ-pipeline-raw-data-dev"
table_path_s3 = f"s3a://{S3_BUCKET}/delta_lake/bea/nipa_observations"

print(f"Reading Delta table directly from S3: {table_path_s3}")

df_bea = spark.read.format("delta").load(table_path_s3)

df_bea.show(5, truncate=False)

## 4. Delta Table History
Delta Lake automatically keeps a transaction log of every operation.

In [ ]:
from delta.tables import DeltaTable

# Access the DeltaTable instance and compute its history directly from S3
delta_table = DeltaTable.forPath(spark, table_path_s3)

# Display the history of the Delta table (the transaction log)
delta_table.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

## 5. Time Travel (VERSION AS OF)
Query older snapshots of the table to avoid look-ahead bias.

In [ ]:
history_df = delta_table.history().collect()

if len(history_df) >= 2:
    latest_version = history_df[0]["version"]
    older_version = history_df[-1]["version"] 
    
    print(f"Latest version: {latest_version}")
    print(f"Older version: {older_version}")
    
    df_older = spark.read.format("delta").option("versionAsOf", older_version).load(table_path_s3)
    
    print(f"Row count at version {older_version}: {df_older.count():,}")
    print(f"Row count at version {latest_version}: {df_bea.count():,}")
else:
    print("Table only has 1 version.")
    
    df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(table_path_s3)
    df_v0.show(5)


## 6. SQL Syntax Example
We can register the table as a temporary view to query it with standard SQL within the PySpark environment.

In [ ]:
df_bea.createOrReplaceTempView("bea_nipa_observations_view")

spark.sql("""
    SELECT period_date, series_name, value, ingested_at
    FROM bea_nipa_observations_view
    ORDER BY ingested_at DESC
    LIMIT 10
""").show()
